# NB_Silver_Marketing_Qualified_Leads — Bronze to Silver Transformation

## Section 1 — Purpose, Business Context & Verified Profile

- **Why MQL First:** `silver.marketing_qualified_leads` is the parent funnel table (`mql_id` PK) referenced by `silver.closed_deals` (`10.53%` conversion rate, `842 / 8,000`). Building MQL first allows `NB_Silver_Closed_Deals.ipynb` to validate `mql_id` referential integrity and `won_date >= first_contact_date` chronology directly against Silver.
- **Source Table:** `Cloud_Retail_Data_Lakehouse_v2.dbo.marketing_qualified_leads` → OneLake Shortcut (`Files/bronze/marketing/marketing_qualified_leads/`).
- **Silver Target:** `silver.marketing_qualified_leads` (Managed Delta Table) + `silver._quarantine_marketing_qualified_leads` (Conditional Quarantine).
- **Grain:** One row per Marketing Qualified Lead (`mql_id`, `8,000` rows).
- **Verified Bronze Profile (`8,000` rows × `4` columns):**
  - `0` duplicate `mql_id`s, `0` invalid 32-char hex IDs (`mql_id` & `landing_page_id`), `495` distinct `landing_page_id`s.
  - `first_contact_date`: `DATE` from `2017-06-14` to `2018-05-31` (`0` NULLs, `0` unparseable).
  - `origin`: **`60` NULLs** + **`1,099` literal `'unknown'`** (`1,159` combined unknown-attribution leads, `14.49%`) + `9` known channels (`organic_search`: `2,296`, `paid_search`: `1,586`, `social`: `1,350`, `direct_traffic`: `499`, `email`: `493`, `referral`: `284`, `other`: `150`, `display`: `118`, `other_publicities`: `65`).

> **WARNING — SAFETY LOCK ENABLED (`WRITE_ENABLED = False`):** Do not enable writes until Dry Run validation passes.

## Section 2 — Configuration & Snapshot Auto-Discovery

Automatically discovers the latest `.parquet` snapshot inside `Files/bronze/marketing/marketing_qualified_leads/` (written by `PL_SQLServer_To_ADLS_Bronze_Metadata_Load`), extracts `INGESTION_RUN_ID` from the filename when present, and locks writes behind `WRITE_ENABLED = False`.

In [ ]:
# Section 2: Configuration & Snapshot Auto-Discovery
from pyspark.sql import functions as F

WRITE_ENABLED = False  # SAFETY LOCK: flip to True only after Dry Run review

BRONZE_FOLDER = "Files/bronze/marketing/marketing_qualified_leads"
parquet_files = sorted([f.name for f in notebookutils.fs.ls(BRONZE_FOLDER) if f.name.endswith(".parquet")])
if not parquet_files:
    raise FileNotFoundError(f"No Parquet snapshots found in {BRONZE_FOLDER}")

LATEST_FILE = parquet_files[-1]
BRONZE_PATH = f"{BRONZE_FOLDER}/{LATEST_FILE}"
INGESTION_RUN_ID = LATEST_FILE.rsplit("_", 1)[-1].replace(".parquet", "") if "_" in LATEST_FILE else "manual_run"

SILVER_SCHEMA = "silver"
SILVER_TARGET_TABLE = f"{SILVER_SCHEMA}.marketing_qualified_leads"
QUARANTINE_TARGET_TABLE = f"{SILVER_SCHEMA}._quarantine_marketing_qualified_leads"

SOURCE_SYSTEM = "sqlserver_olist"
SOURCE_TABLE = "dbo.marketing_qualified_leads"

EXPECTED_BRONZE_COLS = [
    "mql_id",
    "first_contact_date",
    "landing_page_id",
    "origin",
]

print(f"Discovered Parquet snapshots : {len(parquet_files)}")
print(f"Selected BRONZE_PATH         : {BRONZE_PATH}")
print(f"Extracted INGESTION_RUN_ID   : {INGESTION_RUN_ID}")
print(f"Configured SILVER_TARGET     : {SILVER_TARGET_TABLE}")
print(f"WRITE_ENABLED                : {WRITE_ENABLED}")

## Section 3 — Distributed Bronze Parquet Ingestion

Reads the discovered Bronze Parquet snapshot directly with PySpark and attaches `_source_file` lineage.

In [ ]:
# Section 3: Read Bronze Parquet & Inspect Schema
df_bronze_raw = (
    spark.read
    .format("parquet")
    .load(BRONZE_PATH)
    .withColumn("_source_file", F.input_file_name())
)

bronze_count = df_bronze_raw.count()
print(f"Bronze MQL Row Count: {bronze_count:,} (Expected: 8,000)")
df_bronze_raw.printSchema()
display(df_bronze_raw.limit(5))

## Stage 1 — Schema Contract & Raw Null Baseline

Verifies that all `4` expected columns exist, flags any schema drift, and profiles raw `NULL`s (`origin` expected `60` NULLs).

In [ ]:
# Stage 1: Schema Contract & Baseline Profiling
actual_data_cols = [c for c in df_bronze_raw.columns if c != "_source_file"]
missing_cols = [c for c in EXPECTED_BRONZE_COLS if c not in actual_data_cols]
extra_cols = [c for c in actual_data_cols if c not in EXPECTED_BRONZE_COLS]

if missing_cols:
    raise ValueError(f"Stage 1 FAILED — Missing required Bronze columns: {missing_cols}")

if extra_cols:
    print(f"WARNING — Extra columns observed in Bronze (excluded from Silver contract): {extra_cols}")

df_bronze = df_bronze_raw.select(*EXPECTED_BRONZE_COLS, "_source_file")

raw_nulls = df_bronze.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}_nulls")
    for c in EXPECTED_BRONZE_COLS
])
print("Stage 1 Schema Validation: PASSED")
display(raw_nulls)

## Stage 2 — Standardization & Origin Normalization

1. Lowercases and trims `mql_id`, `landing_page_id`, and `origin`.
2. Safe-casts `first_contact_date` to `DATE` via `try_cast`.
3. Normalizes blank/`NULL` `origin` to `NULL` in `origin` while adding `is_unknown_origin` (`origin IS NULL OR origin == 'unknown'`, Expected: `1,159` rows = `60` NULL + `1,099` `'unknown'`).

In [ ]:
# Stage 2: Standardization & Origin Normalization
df_pre = df_bronze.withColumn("_raw_fcd_str", F.col("first_contact_date").cast("string"))

df_std = (
    df_pre
    .withColumn("mql_id", F.lower(F.trim(F.col("mql_id").cast("string"))))
    .withColumn("landing_page_id", F.lower(F.trim(F.col("landing_page_id").cast("string"))))
    .withColumn("first_contact_date", F.expr("try_cast(first_contact_date AS DATE)"))
    .withColumn(
        "origin",
        F.when(F.length(F.trim(F.col("origin").cast("string"))) == 0, F.lit(None).cast("string"))
        .otherwise(F.lower(F.trim(F.col("origin").cast("string"))))
    )
    .withColumn("_fcd_cast_failed", F.col("_raw_fcd_str").isNotNull() & F.col("first_contact_date").isNull())
    .withColumn("is_unknown_origin", F.col("origin").isNull() | (F.col("origin") == F.lit("unknown")))
)

print(f"Stage 2 complete. Cast failures: {df_std.filter(F.col('_fcd_cast_failed')).count()}")
display(df_std.groupBy("origin").count().orderBy(F.desc("count")))

## Stage 3 & 4 — PK Uniqueness, Hex32 Validation & Quarantine Routing

Enforces `1:1` uniqueness on `mql_id`, 32-char lowercase hex format (`^[0-9a-f]{32}$`) on `mql_id` and `landing_page_id`, and non-null `first_contact_date`.

In [ ]:
# Stage 3 & 4: PK Uniqueness, Hex32 Validation & Quarantine Routing
hex32_regex = r"^[0-9a-f]{32}$"
exact_dup_count = bronze_count - df_bronze.dropDuplicates(EXPECTED_BRONZE_COLS).count()

dup_pk_df = (
    df_std.filter(F.col("mql_id").isNotNull())
    .groupBy("mql_id").count().filter(F.col("count") > 1)
    .select("mql_id").withColumn("_dup_pk", F.lit(True))
)
dup_pk_cnt = dup_pk_df.count()

df_checked = (
    df_std
    .join(dup_pk_df, on="mql_id", how="left")
    .withColumn("_dup_pk", F.coalesce(F.col("_dup_pk"), F.lit(False)))
    .withColumn(
        "dq_rejection_reason",
        F.concat_ws(
            " | ",
            F.when(~F.coalesce(F.col("mql_id"), F.lit("")).rlike(hex32_regex), F.lit("INVALID_MQL_ID_HEX32")),
            F.when(~F.coalesce(F.col("landing_page_id"), F.lit("")).rlike(hex32_regex), F.lit("INVALID_LANDING_PAGE_ID_HEX32")),
            F.when(F.col("first_contact_date").isNull(), F.lit("NULL_OR_INVALID_FIRST_CONTACT_DATE")),
            F.when(F.col("_fcd_cast_failed"), F.lit("DATE_CAST_FAILED")),
            F.when(F.col("_dup_pk"), F.lit("DUPLICATE_MQL_ID_PK")),
        )
    )
)

df_valid = df_checked.filter(F.length(F.col("dq_rejection_reason")) == 0)
df_quarantine = (
    df_checked.filter(F.length(F.col("dq_rejection_reason")) > 0)
    .select(*EXPECTED_BRONZE_COLS, "dq_rejection_reason", "_source_file")
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("quarantined_at", F.current_timestamp())
)

valid_count = df_valid.count()
quarantine_count = df_quarantine.count()
print(f"Exact Duplicates : {exact_dup_count} | Duplicate PKs : {dup_pk_cnt}")
print(f"Valid MQL Rows   : {valid_count:,} (Expected: 8,000)")
print(f"Quarantined Rows : {quarantine_count:,} (Expected: 0)")

## Stage 5 — Lineage Enrichment, Diagnostics & Reconciliation

Projects the `9`-column Silver contract (`4` core columns + `is_unknown_origin` + `4` standardized lineage columns) and asserts `bronze_count == silver_count + quarantine_count`.

In [ ]:
# Stage 5: Lineage & Reconciliation
df_silver = (
    df_valid
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("silver_processed_at", F.current_timestamp())
    .select(
        "mql_id",
        "first_contact_date",
        "landing_page_id",
        "origin",
        "is_unknown_origin",
        "source_system",
        "source_table",
        "ingestion_run_id",
        "silver_processed_at",
    )
)

silver_count = df_silver.count()
assert bronze_count == silver_count + quarantine_count, (
    f"Reconciliation mismatch: bronze={bronze_count}, silver={silver_count}, quarantine={quarantine_count}"
)

diag = df_silver.agg(
    F.countDistinct("landing_page_id").alias("distinct_landing_pages"),
    F.sum(F.when(F.col("is_unknown_origin"), 1).otherwise(0)).alias("unknown_origin_rows"),
    F.min("first_contact_date").alias("min_fcd"),
    F.max("first_contact_date").alias("max_fcd"),
).collect()[0]

print(f"Stage 5 Reconciliation PASSED : bronze={bronze_count:,}, silver={silver_count:,}, quarantine={quarantine_count:,}")
print(f"Distinct Landing Pages        : {diag['distinct_landing_pages']} (Expected: 495)")
print(f"Unknown Origin (NULL+unknown) : {diag['unknown_origin_rows']:,} (Expected: 1,159)")
print(f"Date Range                    : {diag['min_fcd']} -> {diag['max_fcd']}")
df_silver.printSchema()

## Section 5 — Guarded Delta Write (`silver.marketing_qualified_leads`)

Creates `silver` schema if missing and persists `silver.marketing_qualified_leads` strictly when `WRITE_ENABLED = True`.

In [ ]:
# Section 5: Guarded Delta Write
if WRITE_ENABLED:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TARGET_TABLE)
    print(f"Wrote {silver_count:,} rows to {SILVER_TARGET_TABLE}")

    if quarantine_count > 0:
        df_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(QUARANTINE_TARGET_TABLE)
        print(f"Wrote {quarantine_count:,} rows to {QUARANTINE_TARGET_TABLE}")
    else:
        print("Zero quarantined rows — skipping quarantine table write.")
else:
    print("WRITE_ENABLED is False — dry validation only; no Silver or Quarantine data was written.")

## Section 6 — Post-Write Delta Read-Back Verification

Reads back `silver.marketing_qualified_leads` when `WRITE_ENABLED = True` and confirms `8,000` rows and `0` duplicate `mql_id`s.

In [ ]:
# Section 6: Post-Write Validation
if WRITE_ENABLED:
    df_check = spark.table(SILVER_TARGET_TABLE)
    chk_count = df_check.count()
    chk_dup = df_check.groupBy("mql_id").count().filter(F.col("count") > 1).count()
    assert chk_count == silver_count, f"Row mismatch: {chk_count} vs {silver_count}"
    assert chk_dup == 0, f"Duplicate mql_id found: {chk_dup}"
    print(f"Post-write validation PASSED — rows={chk_count:,}, dup_pk={chk_dup}")
    display(df_check.limit(5))
else:
    print("Post-write validation skipped because WRITE_ENABLED is False.")